# XG-BOOST:

---

## Importing Libraries:

In [1]:
import time 

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from xgboost import XGBRegressor

## Importing Dataset:

In [2]:
data = pd.read_csv("Cleaned Laptop Dataset.csv")

In [3]:
data.drop(columns=['Unnamed: 0'], inplace=True)

## Building Model:

In [4]:
X = data.drop(columns=['Price'])
y = data['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [5]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series']) 
], remainder='passthrough')

X_train_encoded = preprocessing.fit_transform(X_train)
X_test_encoded = preprocessing.transform(X_test)

In [6]:
param = {'objective' : ['reg:squarederror', 'reg:absoluteerror'],
        'n_estimators' : [500, 600, 700, 800],
        'max_depth' : [5, 6, 7, 8],
        'learning_rate' : [0.1, 0.01]}

grid = GridSearchCV(
    estimator=XGBRegressor(),
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid.fit(X_train_encoded, y_train)

print(f"Best Parameters: {grid.best_params_}")
print(f"Best Score: {grid.best_score_ * 100:.2f}%")

Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 700, 'objective': 'reg:absoluteerror'}
Best Score: 83.71%


In [7]:
param = {'reg_alpha' : [0, 0.1, 1, 10, 100],
        'reg_lambda' : [0, 0.1, 1, 10, 100]}

grid = GridSearchCV(
    estimator=XGBRegressor(objective='reg:absoluteerror', n_estimators=700, max_depth=7, learning_rate=0.1),
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid.fit(X_train_encoded, y_train)

print(f"Best Parameters: {grid.best_params_}")
print(f"Best Score: {grid.best_score_ * 100:.2f}%")

Best Parameters: {'reg_alpha': 10, 'reg_lambda': 1}
Best Score: 84.21%


In [8]:
xgb = XGBRegressor(objective='reg:absoluteerror',learning_rate=0.1, max_depth=7, n_estimators=700, reg_alpha=10, reg_lambda=1)

start_time = time.time()
xgb.fit(X_train_encoded, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 3.83s


## Results:

In [9]:
def adjusted_r2_score(y, y_pred):

    r2 = r2_score(y, y_pred)
    n = X.shape[0]
    k = X.shape[1]

    return 1 - ((1 - r2) * (n - 1) / (n - 1 - k))

In [10]:
# Results on Training Data:

y_pred_train = xgb.predict(X_train_encoded)

print(f"r2 Score (Trainig Data): {r2_score(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nmean absolute error(Training Data): {mean_absolute_percentage_error(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_train, y_pred_train) * 100 :.2f}%")

r2 Score (Trainig Data): 93.24%

mean absolute error(Training Data): 8.35%

adjusted r2 Score: 93.23%


In [11]:
# Results on Testing Data:

y_pred_test = xgb.predict(X_test_encoded)

print(f"r2 Score (Testing Data): {r2_score(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nmean absolute error (Testing Data): {mean_absolute_percentage_error(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_test, y_pred_test) * 100 :.2f}%")

r2 Score (Testing Data): 87.45%

mean absolute error (Testing Data): 14.98%

adjusted r2 Score: 87.42%


## Cross-Val-Score:

In [12]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series']) 
], remainder='passthrough')

xgb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', XGBRegressor(objective='reg:absoluteerror',learning_rate=0.1, max_depth=7, n_estimators=700, reg_alpha=10, reg_lambda=1))
])

print(f"r2-Score: {np.mean(cross_val_score(estimator=xgb_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100:.2f}%")

print(f"mean absolute error: {-np.mean(cross_val_score(estimator=xgb_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100:.2f}%")

r2-Score: 85.14%
mean absolute error: 15.33%


---

# Experimentaion:

- ## Transformation (target feature):

In [13]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

xgb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', XGBRegressor())
])

xgb_model = TransformedTargetRegressor(
    regressor=xgb_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param = {'regressor__model__objective' : ['reg:squarederror', 'reg:absoluteerror'],
        'regressor__model__n_estimators' : [600, 700, 800, 900, 1000],
        'regressor__model__max_depth' : [5, 6, 7, 8],
        'regressor__model__learning_rate' : [0.1, 0.01]}

grid_model = GridSearchCV(
    estimator=xgb_model,
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid_model.fit(X_train, y_train)

print(f"Best Parameters: {grid_model.best_params_}")

Best Parameters: {'regressor__model__learning_rate': 0.01, 'regressor__model__max_depth': 6, 'regressor__model__n_estimators': 1000, 'regressor__model__objective': 'reg:squarederror'}


In [15]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

xgb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', XGBRegressor(objective='reg:squarederror', n_estimators=1000, max_depth=6, learning_rate=0.01))
])

xgb_model = TransformedTargetRegressor(
    regressor=xgb_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param = {'regressor__model__reg_alpha' : [0, 0.1, 1, 10, 100],
        'regressor__model__reg_lambda' : [0, 0.1, 1, 10, 100]}

grid_model = GridSearchCV(
    estimator=xgb_model,
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid_model.fit(X_train, y_train)

print(f"Best Parameters: {grid_model.best_params_}")

Best Parameters: {'regressor__model__reg_alpha': 0, 'regressor__model__reg_lambda': 1}


In [17]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

xgb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', XGBRegressor(learning_rate=0.01, max_depth=6, n_estimators=1000, objective='reg:squarederror', reg_alpha=0, reg_lambda=1))
])

xgb_model = TransformedTargetRegressor(
    regressor=xgb_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print(f"r2 Score: {np.mean(cross_val_score(estimator=xgb_model, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=xgb_model, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 84.07%

mean absolute error: 15.89%


# Conclusion:

- *XGBoost* performs overall best, compared to all algorithms with an **r2-Score = 85.14% & M.A.P.E = 15.33%**.